# Interactive exploratory visualization
Interactive visualizations that analyze the
evolution of visa-free access to the US.

# DATASET

In [1]:
import pprint
import pydytuesday
import pandas
from IPython.display import display
import json
import altair as alt

country_lists = pandas.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-09/country_lists.csv')
rank_by_year = pandas.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-09/rank_by_year.csv')

print("Country Lists Dataset")
display(country_lists.head())

print("Rank by Year Dataset")
display(rank_by_year.head())

andorra_visa_required = country_lists.loc[country_lists['country'] == 'Andorra', 'visa_required'].values[0]
andorra_visa_required_json = json.loads(andorra_visa_required)
print("Andorra citizens require visa to enter:")
# Print only the first 3 countries for brevity
pprint.pprint(andorra_visa_required_json[0][:3])
andorra_visa_free_access = country_lists.loc[country_lists['country'] == 'Andorra', 'visa_free_access'].values[0]
andorra_visa_free_access_json = json.loads(andorra_visa_free_access)
visa_free_access_count = len(andorra_visa_free_access_json[0])
print(f"Number of countries that can enter Andorra with visa-free access: {visa_free_access_count}")

print(f"Total number of countries in the dataset: {country_lists.shape[0]}")

Country Lists Dataset


,code,country,visa_required,visa_online,visa_on_arrival,visa_free_access,electronic_travel_authorisation
0,PS,Palestinian Territory,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AG"",""name"":""Antigua and Barbuda""},{...","[[{""code"":""BD"",""name"":""Bangladesh""},{""code"":""B...","[[{""code"":""BO"",""name"":""Bolivia""},{""code"":""CK"",...","[[{""code"":""LK"",""name"":""Sri Lanka""},{""code"":""KE..."
1,AD,Andorra,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AO"",""name"":""Angola""},{""code"":""AZ"",""...","[[{""code"":""BH"",""name"":""Bahrain""},{""code"":""BD"",...","[[{""code"":""JP"",""name"":""Japan""},{""code"":""AL"",""n...","[[{""code"":""AU"",""name"":""Australia""},{""code"":""CA..."
2,VA,Vatican City,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AZ"",""name"":""Azerbaijan""},{""code"":""B...","[[{""code"":""BH"",""name"":""Bahrain""},{""code"":""BD"",...","[[{""code"":""AL"",""name"":""Albania""},{""code"":""AD"",...","[[{""code"":""AU"",""name"":""Australia""},{""code"":""CA..."
3,SM,San Marino,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AZ"",""name"":""Azerbaijan""},{""code"":""B...","[[{""code"":""BH"",""name"":""Bahrain""},{""code"":""BD"",...","[[{""code"":""JP"",""name"":""Japan""},{""code"":""AL"",""n...","[[{""code"":""AU"",""name"":""Australia""},{""code"":""CA..."
4,MC,Monaco,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AZ"",""name"":""Azerbaijan""},{""code"":""B...","[[{""code"":""BH"",""name"":""Bahrain""},{""code"":""BD"",...","[[{""code"":""JP"",""name"":""Japan""},{""code"":""AL"",""n...","[[{""code"":""AU"",""name"":""Australia""},{""code"":""CA..."


Rank by Year Dataset


,code,country,region,rank,visa_free_count,year
0,AF,Afghanistan,ASIA,116,26,2021
1,AF,Afghanistan,ASIA,106,26,2020
2,AF,Afghanistan,ASIA,106,30,2018
3,AF,Afghanistan,ASIA,104,24,2017
4,AF,Afghanistan,ASIA,104,25,2016


Andorra citizens require visa to enter:
[{'code': 'AF', 'name': 'Afghanistan'},
 {'code': 'DZ', 'name': 'Algeria'},
 {'code': 'BT', 'name': 'Bhutan'}]
Number of countries that can enter Andorra with visa-free access: 120
Total number of countries in the dataset: 199


## Dataset Analysis

We observe a dataset with 7 entries. Most of them have string values that contain a json list.

For each country we will have a list that contains the relation of this country with the rest of countris. Lets visualize an example:
Andorra citizens require visa to enter:
[[{'code': 'AF', 'name': 'Afghanistan'},
  {'code': 'DZ', 'name': 'Algeria'},
  {'code': 'BT', 'name': 'Bhutan'},
  {'code': 'BN', 'name': 'Brunei'},
  ....
]]

To be able to make queryes on an efficient way we should serialize the jsons.

## Dataset Cleaning

### Add ISO 3166-1 codes to each country

In [2]:
import pycountry

# Create a function to get ISO 3166-1 numeric code from country name
def get_iso_numeric_code(country_name):
    try:
        # Try direct lookup
        country = pycountry.countries.get(name=country_name)
        if country:
            return int(country.numeric)
        
        # Try fuzzy search
        country = pycountry.countries.search_fuzzy(country_name)[0]
        return int(country.numeric)
    except:
        return None

# Add ISO 3166-1 numeric codes to rank_by_year
rank_by_year['iso_numeric'] = rank_by_year['country'].apply(get_iso_numeric_code).astype('Int64')

# Check results
print(f"Countries with ISO codes: {rank_by_year['iso_numeric'].notna().sum()}")
print(f"Countries without ISO codes: {rank_by_year['iso_numeric'].isna().sum()}")

# Display countries without codes for manual mapping if needed
missing_codes = rank_by_year[rank_by_year['iso_numeric'].isna()]['country'].unique()
if len(missing_codes) > 0:
    print("\nCountries without ISO codes:")
    print(missing_codes)

display(rank_by_year[['country', 'iso_numeric']].drop_duplicates())

Countries with ISO codes: 3720
Countries without ISO codes: 230

Countries without ISO codes:
['Cape Verde Islands' 'Comoro Islands' 'Congo (Rep.)' 'Congo (Dem. Rep.)'
 'Hong Kong (SAR China)' 'Macao (SAR China)' 'Palau Islands'
 'St. Kitts and Nevis' 'St. Lucia' 'St. Vincent and the Grenadines'
 'Taiwan (Chinese Taipei)' 'Palestinian Territory']


,country,iso_numeric
0,Afghanistan,4
20,Albania,8
40,Algeria,12
60,Angola,24
80,Antigua and Barbuda,28
...,...,...
3854,Monaco,492
3874,San Marino,674
3894,Vatican City,336
3912,Andorra,20


# Q1: 1. Which continent has the most visa-free destinations? How does this vary when considering population size?

## most visa-free destinations
We have 7 continents and 20 different years. My first representation use "x" axis as time while "y" axis would be  visa free count. Each continent would be encoded with diferent colors so i will use a categorized color palete.

In [3]:
print(rank_by_year['region'].unique())
print(rank_by_year['year'].unique())

['ASIA' 'EUROPE' 'AFRICA' 'CARIBBEAN' 'AMERICAS' 'MIDDLE EAST' 'OCEANIA']
[2021 2020 2018 2017 2016 2015 2014 2013 2012 2011 2010 2009 2008 2007
 2006 2019 2022 2023 2024 2025]


In [4]:

# Altair line chart: visa-free count over time by continent
chart = alt.Chart(rank_by_year).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('visa_free_count:Q', title='Visa-Free Destinations'),
    color=alt.Color('region:N', title='Continent'),
    tooltip=['country', 'region', 'year', 'visa_free_count']
).properties(
    title='Visa-Free Destinations by Continent Over Time',
    width=700,
    height=400
).interactive()

chart

alt.Chart(...)

The obvious problem with this plot is that each continent has many countries and each country has a diferent number of "visa free count". We need some metric to compute how easy is to enter the continent from any other country. That means we have a set of N countries C={country1,country2,...,countryN}, we select Cn where 0<n<|C| and iterate over the rest of the countries to check if they can enter Cn, store the value in Fn, and once we have computed F (count of countries that can enter Cn) of each country, we add up all Fn based on the continent they belong to.

This can be done using Altair by adding up all country counts based on the continent. 

In [5]:
# Group by region and year, sum visa_free_count for each continent per year
continent_year_sum = rank_by_year.groupby(['region', 'year'], as_index=False)['visa_free_count'].sum()

# Altair line chart: total visa-free count over time by continent
continent_chart = alt.Chart(continent_year_sum).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('visa_free_count:Q', title='Total Visa-Free Destinations'),
    color=alt.Color('region:N', title='Continent'),
    tooltip=['region', 'year', 'visa_free_count']
).properties(
    title='Total Visa-Free Destinations by Continent Over Time',
    width=700,
    height=400
).interactive()

continent_chart

alt.Chart(...)

There are some missing years [2007, 2009] that can be just ignored. The plot will linearly interpolate this values but it will not suppose a big problem since it is not alterating the real data.

In [6]:
# Remove years 2007 and 2009 with zero counts
all_years = sorted(continent_year_sum['year'].unique())
filtered_continent_year_sum = continent_year_sum[~continent_year_sum['year'].isin([2007, 2009])]

# Altair line chart: total visa-free count over time by continent (filtered)
filtered_continent_chart = alt.Chart(filtered_continent_year_sum).mark_line(point=True).encode(
    x=alt.X('year:Q', title='Year', axis=alt.Axis(values=all_years) ),
    y=alt.Y('visa_free_count:Q', title='Total Visa-Free Destinations'),
    color=alt.Color('region:N', title='Continent'),
    tooltip=['region', 'year', 'visa_free_count']
).properties(
    title='Total Visa-Free Destinations by Continent Over Time (Filtered)',
    width=700,
    height=400
).interactive()

filtered_continent_chart

alt.Chart(...)

This graph can be triky to interpret since we do not have 8000 countirs in europe. The thing is that there are many countries that can go to certain europe countries without visa. For example, there are 120 countries that can enter andorra, in case it is the same for spane we end up with 240 countries that canenter europe, but in reality, most of this countries will be repeated and in truth the real amount of countries that can enter europe are 123 for example. 

In [13]:
# Interactive strip plot with year slider
year_slider = alt.binding_range(min=2006, max=2025, step=1, name='Year: ')
year_select = alt.selection_point(name='year_selection', fields=['year'], bind=year_slider, value=2021)

strip_chart = alt.Chart(rank_by_year).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('region:N', title='Continent'),
    y=alt.Y('visa_free_count:Q', title='Visa-Free Destinations', scale=alt.Scale(domain=[0, 200])),
    color=alt.Color('region:N', title='Continent'),
    tooltip=['country', 'region', 'year', 'visa_free_count'],
    xOffset=alt.XOffset('jitter:Q', scale=alt.Scale(domain=[-0.2, 0.2]))
).add_params(
    year_select
).transform_filter(
    year_select
).transform_calculate(
        jitter="((indexof('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz', substring(datum.country, 0, 1)) % 20) / 100 - 0.1)"

).properties(
    title='Visa-Free Destinations by Continent (Select Year)',
    width=700,
    height=400
)

mean_line = alt.Chart(rank_by_year).mark_rule(
    color='black',
    strokeDash=[4,2]
).encode(
    x=alt.X('region:N'),
    y='mean(visa_free_count):Q',
    tooltip=[alt.Tooltip('region:N'), alt.Tooltip('mean(visa_free_count):Q', title='Mean')],
).add_params(
    year_select
).transform_filter(
    year_select
)

# Combine strip plot and mean lines
strip_chart + mean_line

alt.LayerChart(...)

In [8]:
from vega_datasets import data

# Create a geospatial map showing visa-free count per country using Altair

# Load world topojson from Vega datasets
world_map = alt.topo_feature(data.world_110m.url, 'countries')

# Prepare country data for 2021
country_counts = rank_by_year[rank_by_year['year'] == 2025][['country', 'visa_free_count', 'iso_numeric']]
print(country_counts.head())
# Altair geoshape map
geo_chart = alt.Chart(world_map).mark_geoshape(
    stroke='black',
    strokeWidth=0.5
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(data=country_counts, key='iso_numeric', fields = ['iso_numeric', 'visa_free_count'])
).encode(
    tooltip=['id:N', 'iso_numeric:N', 'visa_free_count:Q'],
    color=alt.Color('visa_free_count:Q', title='Visa-Free Count', scale=alt.Scale(scheme='blues'))
).project(
    type='naturalEarth1'
).properties(
    title='Visa-Free Count per Country (2021)',
    width=900,
    height=500
)

geo_chart

                country  visa_free_count  iso_numeric
19          Afghanistan               25            4
39              Albania              123            8
59              Algeria               55           12
79               Angola               48           24
99  Antigua and Barbuda              152           28


alt.Chart(...)

In [9]:
# Check country names in your dataset
print("Countries in your dataset (first 10):")
print(sorted(rank_by_year['country'].unique())[:10])

# Check country names in the world map
from vega_datasets import data
import json
import requests

world_data = json.loads(requests.get(data.world_110m.url).text)
print(world_data.keys())
print(world_data['objects'].keys())
print(world_data['objects']['countries'])
# Extract country names from the geometries
map_countries = []
for feature in world_data['objects']['countries']['geometries']:
    if 'properties' in feature and feature['properties']:
        # Try different possible name fields
        name = feature['properties'].get('name') or feature['properties'].get('NAME') or feature['properties'].get('admin')
        if name:
            map_countries.append(name)

print("\nCountries in world map (first 10):")
print(sorted(map_countries)[:10])

# Also check what properties are available
print("\nExample properties from world map:")
for feature in world_data['objects']['countries']['geometries'][:3]:
    if 'properties' in feature:
        print(feature['properties'])

# Check for matches
your_countries = set(rank_by_year['country'].unique())
map_country_set = set(map_countries)

print(f"\nMatching countries: {len(your_countries & map_country_set)}")
print(f"Countries only in your data: {len(your_countries - map_country_set)}")
print(f"\nSome matched countries:")
print(list(your_countries & map_country_set)[:10])
print(f"\nSome unmatched countries from your data:")
print(list(your_countries - map_country_set)[:10])

Countries in your dataset (first 10):
['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria']
dict_keys(['type', 'transform', 'objects', 'arcs'])
dict_keys(['land', 'countries'])
{'type': 'GeometryCollection', 'geometries': [{'type': 'Polygon', 'arcs': [[499, 500, 501, 502, 503, 504]], 'id': 4}, {'type': 'MultiPolygon', 'arcs': [[[505, 506, 352, 507]], [[354, 508, 509]]], 'id': 24}, {'type': 'Polygon', 'arcs': [[510, 511, 414, 512, 513, 514]], 'id': 8}, {'type': 'Polygon', 'arcs': [[312, 515, 314, 516, 517]], 'id': 784}, {'type': 'MultiPolygon', 'arcs': [[[518, 11]], [[519, 520, 521, 166, 522, 168, 523, 524]]], 'id': 32}, {'type': 'Polygon', 'arcs': [[525, 526, 527, 528, 529]], 'id': 51}, {'type': 'MultiPolygon', 'arcs': [[[0]], [[1]], [[2]], [[3]], [[4]], [[5]], [[6]], [[530, 531]]], 'id': 10}, {'type': 'Polygon', 'arcs': [[13]], 'id': 260}, {'type': 'MultiPolygon', 'arcs': [[[14]], [[24]]], 'id': 36}, {'type': 

## Summary
This graph last graph aims to give a general idea of how permissive is a continent and not which countries are allowed to enter other country or continent. We can observe that in 2025, almost all europe countries allow around 180 countries to enter with visa-free. Africa's countries are way more restrictive allowing only around 60 countries to enter.

We dont show how many different countries can eneter africa though. it could be that tow african countries allow complitelly different citizenships to enter which could be relevant for further analysis.

## Population Size

TODO: I dont have population size and i do not know if it is for country or continent

# Q2: Which countries have experienced the greatest changes as visa-free countries between 2006 and 2021?

We are request to show the visa-free changes on each country. The problem is that we are working with 199 countries and 15 years (15 categorized samples). Showing so many values in a single chart will only produce visaul noise. 

Since we are interested in showing only the top 5 or so, we will create a interactive integer inputs to set the rendered range (eg: from 0 to 5, from 12 to 18, from 0 to 10...). We will use a categorized palette. It will be a bar chart sinze "x" axis is not continuous. each country will have two bars, 2006 and 2021, to compare if it has increased or decreased. We will place the bars horizontally to be able to scale it in case we have many countries for rendering.

We are not only interested in the increment and decrement of new free-visa access, it is also relevant to know how much they had before and after. That is why we are not using a single bar (that can take positive and negative values) to represent the change.

In [10]:
# Calculate change in visa-free count between 2006 and 2021
data_2006 = rank_by_year[rank_by_year['year'] == 2006][['country', 'visa_free_count', 'region']].rename(columns={'visa_free_count': 'count_2006'})
data_2021 = rank_by_year[rank_by_year['year'] == 2021][['country', 'visa_free_count']].rename(columns={'visa_free_count': 'count_2021'})

# Merge and calculate change
changes = data_2006.merge(data_2021, on='country')
changes['change'] = changes['count_2021'] - changes['count_2006']
changes = changes.sort_values('change', ascending=False).reset_index(drop=True)
changes['rank'] = changes.index

# Prepare data for visualization (melt to long format)
changes_long = changes.melt(
    id_vars=['country', 'region', 'change', 'rank'],
    value_vars=['count_2006', 'count_2021'],
    var_name='year',
    value_name='visa_free_count'
)
changes_long['year'] = changes_long['year'].map({'count_2006': '2006', 'count_2021': '2021'})

# Create range sliders for selecting countries
start_slider = alt.binding_range(min=0, max=len(changes)-10, step=1, name='Start Rank: ')
end_slider = alt.binding_range(min=10, max=len(changes), step=1, name='End Rank: ')
start_select = alt.selection_point(name='start', fields=['start'], bind=start_slider, value=0)
end_select = alt.selection_point(name='end', fields=['end'], bind=end_slider, value=10)

# Horizontal bar chart
base = alt.Chart(changes_long).encode(
    y=alt.Y('country:N', title='Country', sort=alt.EncodingSortField(field='change', order='descending'), axis=alt.Axis(labelLimit=200)),
    x=alt.X('visa_free_count:Q', title='Visa-Free Destinations'),
    color=alt.Color('year:N', title='Year', scale=alt.Scale(scheme='category10')),
    yOffset=alt.YOffset('year:N'),
    tooltip=['country', 'region', 'year', 'visa_free_count', 'change']
).transform_calculate(
    start="0",
    end="10"
).add_params(
    start_select,
    end_select
).transform_filter(
    alt.datum.rank >= start_select.start
).transform_filter(
    alt.datum.rank < end_select.end
)

bars = base.mark_bar()

text_labels = base.transform_filter(
    alt.datum.year == '2021'
).mark_text(align='left', dx=3).encode(
    text=alt.Text('change:Q', format='+d')
)

# for some reason i loose the sorting when i add the text
# chart = (bars + text_labels).properties(
chart = (bars).properties(
    title='Countries with Greatest Changes in Visa-Free Access (2006-2021)',
    width=600,
    height=400
)

chart

alt.Chart(...)

This can be further simplifyed to a slope chart to:
1. Add the integer value of change
2. make the chart slimer


In [11]:
# Slope chart: visa_free_count (2006 vs 2021) on x-axis
slope_chart = alt.Chart(changes_long).encode(
    y=alt.Y('country:N', title='Country', sort=alt.EncodingSortField(field='change', order='descending'), axis=alt.Axis(labelLimit=200)),
    x=alt.X('visa_free_count:Q', title='Visa-Free Destinations'),
    color=alt.Color('region:N', title='Continent', scale=alt.Scale(scheme='category10')),
    detail='country:N',
    tooltip=['country', 'region', 'year', 'visa_free_count', 'change']
).transform_calculate(
    start="0",
    end="10"
).add_params(
    start_select,
    end_select
).transform_filter(
    alt.datum.rank >= start_select.start
).transform_filter(
    alt.datum.rank < end_select.end
).properties(
    title='Slope Chart: Visa-Free Access (2006 vs 2021)',
    width=600,
    height=400
)

lines = slope_chart.mark_line(point=True, size=3)

# Text labels only for the change value, positioned at the left
text_labels = slope_chart.transform_filter(
    alt.datum.year == '2021'
).mark_text(align='left', dx=5).encode(
    text=alt.Text('change:Q', format='+d')
)

lines + text_labels

alt.LayerChart(...)

# Q3: What was the impact of COVID-19 on visa-free mobility?

In [12]:
# Analyze the impact of COVID-19 on visa-free mobility
# COVID-19 pandemic started in early 2020, so we'll compare 2019, 2020, 2021, and 2022

# Filter data for the COVID period
covid_years = [2019, 2020, 2021, 2022, 2023]
covid_data = rank_by_year[rank_by_year['year'].isin(covid_years)]

# Calculate average visa-free count by region and year
covid_impact = covid_data.groupby(['region', 'year'], as_index=False)['visa_free_count'].mean()

# Create line chart showing average visa-free access during COVID period
covid_line_chart = alt.Chart(covid_impact).mark_line(point=True, size=3).encode(
    x=alt.X('year:O', title='Year', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('visa_free_count:Q', title='Average Visa-Free Destinations'),
    color=alt.Color('region:N', title='Continent', scale=alt.Scale(scheme='category10')),
    tooltip=['region', 'year', alt.Tooltip('visa_free_count:Q', format='.1f')]
).properties(
    title='Impact of COVID-19 on Visa-Free Mobility by Continent (2019-2023)',
    width=700,
    height=400
)

# Add annotation for COVID-19 start
covid_annotation = alt.Chart(pandas.DataFrame({'x': [2020]})).mark_rule(
    color='red',
    strokeDash=[5, 5],
    size=2
).encode(
    x='x:O'
)

covid_text = alt.Chart(pandas.DataFrame({'x': [2020], 'y': [covid_impact['visa_free_count'].max()]})).mark_text(
    align='left',
    dx=5,
    dy=-10,
    color='red',
    fontWeight='bold'
).encode(
    x='x:O',
    y='y:Q',
    text=alt.value('COVID-19 Pandemic')
)

# Calculate year-over-year changes for each country
covid_country_changes = covid_data.pivot_table(
    index=['country', 'region'],
    columns='year',
    values='visa_free_count'
).reset_index()

# Calculate changes between consecutive years
if 2019 in covid_country_changes.columns and 2020 in covid_country_changes.columns:
    covid_country_changes['change_2019_2020'] = covid_country_changes[2020] - covid_country_changes[2019]
if 2020 in covid_country_changes.columns and 2021 in covid_country_changes.columns:
    covid_country_changes['change_2020_2021'] = covid_country_changes[2021] - covid_country_changes[2020]
if 2021 in covid_country_changes.columns and 2022 in covid_country_changes.columns:
    covid_country_changes['change_2021_2022'] = covid_country_changes[2022] - covid_country_changes[2021]

# Create a small multiples chart showing distribution of changes
change_data = []
for col in ['change_2019_2020', 'change_2020_2021', 'change_2021_2022']:
    if col in covid_country_changes.columns:
        temp = covid_country_changes[['country', 'region', col]].copy()
        temp = temp.dropna()
        temp['period'] = col.replace('change_', '').replace('_', ' → ')
        temp['change'] = temp[col]
        change_data.append(temp[['country', 'region', 'period', 'change']])

change_df = pandas.concat(change_data, ignore_index=True)

# Histogram showing distribution of changes
change_histogram = alt.Chart(change_df).mark_bar(opacity=0.7).encode(
    x=alt.X('change:Q', bin=alt.Bin(maxbins=30), title='Change in Visa-Free Destinations'),
    y=alt.Y('count():Q', title='Number of Countries'),
    color=alt.Color('period:N', title='Period'),
    column=alt.Column('period:N', title='Time Period', header=alt.Header(labelOrient='bottom'))
).properties(
    width=200,
    height=300,
    title='Distribution of Changes in Visa-Free Access During COVID-19'
)

# Display all charts
display(covid_line_chart + covid_annotation + covid_text)
display(change_histogram)

# Summary statistics
print("\nSummary Statistics:")
print(f"Average change 2019→2020: {change_df[change_df['period']=='2019 → 2020']['change'].mean():.2f}")
print(f"Average change 2020→2021: {change_df[change_df['period']=='2020 → 2021']['change'].mean():.2f}")
print(f"Average change 2021→2022: {change_df[change_df['period']=='2021 → 2022']['change'].mean():.2f}")

alt.LayerChart(...)

alt.Chart(...)


Summary Statistics:
Average change 2019→2020: 1.00
Average change 2020→2021: 0.25
Average change 2021→2022: 0.27


# Q4: Do countries that belong to certain global alliances, or have stronger economies, tend to enjoy greater visa-free access?